In [1]:
import numpy as np
import pandas as pd

hosts = {
    1924: 'FRA',
    1928: 'NED',
    1932: 'USA',
    1936: 'GER',
    1948: 'GBR',
    1952: 'FIN',
    1956: 'AUS',
    1960: 'ITA',
    1964: 'JPN',
    1968: 'MEX',
    1972: 'FRG',
    1976: 'CAN',
    1988: 'KOR',
    1992: 'ESP',
    1996: 'USA', 
    2000: 'AUS', 
    2004: 'GRE', 
    2008: 'CHN', 
    2012: 'GBR', 
    2016: 'BRA',
    2020: 'JPN',
    2024: 'FRA'
}

In [2]:
df = pd.read_csv('./data/athlete_events.csv')
df_summer_modern = df[(df['Season']=='Summer') & (df['Year'] != 1980) & (df['Year'] != 1984) & (df['Year'] >= 1924)]
df_summer_modern = df_summer_modern.drop_duplicates(subset=['NOC', 'Games', 'Sport', 'Event', 'Medal'])

df_summer_modern['Medal'] = df_summer_modern['Medal'].notna().astype(int)

df_summer_modern = df_summer_modern.groupby(['NOC', 'Year'])['Medal'].sum().reset_index()

df_last_years = pd.read_csv('data/country-medals-by-year_missing_2018-2026.csv')
df_last_years = df_last_years[df_last_years['Season']=='Summer']
df_last_years = df_last_years.drop(columns=['Gold', 'Silver', 'Bronze'])
df_last_years = df_last_years.drop(columns=['Season'])
df_last_years = df_last_years.rename(columns={'Total_Medals' : 'Medal'})

df_olympics = pd.concat([df_summer_modern, df_last_years], ignore_index=True)

In [3]:
medaglie_in_palio = df_olympics.groupby('Year')['Medal'].sum()

In [4]:
def check_host(row):
    if row['Year'] in hosts and row['NOC'] == hosts[row['Year']]:
        return 1
    else:
        return 0

df_olympics['is_host'] = df_olympics.apply(check_host, axis=1)
df_olympics.head()

,NOC,Year,Medal,is_host
0,AFG,1936,0,0
1,AFG,1948,0,0
2,AFG,1956,0,0
3,AFG,1960,0,0
4,AFG,1964,0,0


In [5]:
df_olympics['Percentage'] = (df_olympics['Medal'] / df_olympics.groupby('Year')['Medal'].transform('sum')) * 100

In [6]:
host_list = list(hosts.values())
df_olympics_hosts = df_olympics[df_olympics['NOC'].isin(host_list)]
df_olympics_hosts.head()

,NOC,Year,Medal,is_host,Percentage
112,AUS,1924,6,0,1.534527
113,AUS,1928,4,0,1.123596
114,AUS,1932,5,0,1.351351
115,AUS,1936,1,0,0.236967
116,AUS,1948,13,0,2.961276


In [7]:
medal_host_and_non_host = []

for noc in host_list:
    medal_host = df_olympics_hosts[(df_olympics_hosts['NOC']==noc) & (df_olympics_hosts['is_host']==1)]['Medal'].sum()
    medal_non_host = df_olympics_hosts[(df_olympics_hosts['NOC']==noc) & (df_olympics_hosts['is_host']==0)]['Medal'].sum()

    medal_host_and_non_host.append({
        'NOC' : noc,
        'Medal_Host' : medal_host,
        'Medal_Non_Host' : medal_non_host
    })

df_medal_host_and_non_host = pd.DataFrame(medal_host_and_non_host)
df_medal_host_and_non_host.head()

,NOC,Medal_Host,Medal_Non_Host
0,FRA,104,489
1,NED,23,302
2,USA,211,1869
3,GER,101,654
4,GBR,92,566


In [8]:
df_medal_host_and_non_host = df_medal_host_and_non_host.drop_duplicates(subset='NOC')

In [9]:
host_list_clean = list(dict.fromkeys(host_list))

in_palio = []

for noc in host_list_clean:
    host_years = []
    for year in hosts:
        if hosts[year] == noc:
            host_years.append(year)
    medaglie_in_palio_host = medaglie_in_palio[medaglie_in_palio.index.isin(host_years)].sum() - df_medal_host_and_non_host[df_medal_host_and_non_host['NOC']==noc]['Medal_Host']
    medaglie_in_palio_non_host = medaglie_in_palio[~medaglie_in_palio.index.isin(host_years)].sum() - df_medal_host_and_non_host[df_medal_host_and_non_host['NOC']==noc]['Medal_Host']

    in_palio.append({
        'NOC' : noc,
        'Palio_Host' : medaglie_in_palio_host.values[0],
        'Palio_Non_Host' : medaglie_in_palio_non_host.values[0]
    })

df_in_palio = pd.DataFrame(in_palio)
df_in_palio.head()

,NOC,Palio_Host,Palio_Non_Host
0,FRA,1331,13325
1,NED,333,14485
2,USA,1000,13442
3,GER,321,14341
4,GBR,1309,13371


In [10]:
df_in_palio = df_in_palio.drop_duplicates(subset='NOC')
df_chi_quadro = pd.merge(df_medal_host_and_non_host, df_in_palio, on='NOC')

In [11]:
def get_contingency_noc(df, noc_name):
    row = df[df['NOC'] == noc_name].iloc[0]
    
    data = {
        'Medals': [row['Medal_Host'], row['Medal_Non_Host']],
        'Palio':  [row['Palio_Host'], row['Palio_Non_Host']]
    }
    
    return pd.DataFrame(data, index=['Host', 'Non-Host'])

tabelle_noc = {}

for noc in host_list_clean:
    tabelle_noc[noc] = get_contingency_noc(df_chi_quadro, noc)

In [12]:
from scipy.stats import chi2_contingency


for noc in tabelle_noc:
    chi2, p, dof, expected = chi2_contingency(tabelle_noc[noc])
    print('CHI QUADRO ', noc)
    print(f"Statistica Chi-Quadro: {chi2:.4f}")
    print(f"P-value: {p:.4f}")
    print(f"Gradi di libertà: {dof}")
    print("\nFrequenze attese (se non ci fosse correlazione):")
    print(expected)
    print('-' * 33)

CHI QUADRO  FRA
Statistica Chi-Quadro: 46.8217
P-value: 0.0000
Gradi di libertà: 1

Frequenze attese (se non ci fosse correlazione):
[[   55.80398715  1379.19601285]
 [  537.19601285 13276.80398715]]
---------------------------------
CHI QUADRO  NED
Statistica Chi-Quadro: 30.2442
P-value: 0.0000
Gradi di libertà: 1

Frequenze attese (se non ci fosse correlazione):
[[7.64049396e+00 3.48359506e+02]
 [3.17359506e+02 1.44696405e+04]]
---------------------------------
CHI QUADRO  USA
Statistica Chi-Quadro: 27.2811
P-value: 0.0000
Gradi di libertà: 1

Frequenze attese (se non ci fosse correlazione):
[[  152.45611911  1058.54388089]
 [ 1927.54388089 13383.45611911]]
---------------------------------
CHI QUADRO  GER
Statistica Chi-Quadro: 333.4070
P-value: 0.0000
Gradi di libertà: 1

Frequenze attese (se non ci fosse correlazione):
[[   20.66614776   401.33385224]
 [  734.33385224 14260.66614776]]
---------------------------------
CHI QUADRO  GBR
Statistica Chi-Quadro: 18.8593
P-value: 0.0000
